# 12. Признаки команды и попадание в топ-10%

Описательно сравниваем частоту top-10 результатов по размеру команды, прошлому опыту и посылкам за первые семь дней. Анализ не устанавливает причинность.

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.build_dataset import get_dataset

df = get_dataset()
df["team_size_band"] = df["team_size"].clip(upper=5).astype(int).astype(str).replace({"5":"5+"})
df["experience_band"] = pd.cut(df["max_prev_comps"], [-1, 0, 2, 5, 10, float("inf")],
    labels=["0", "1-2", "3-5", "6-10", "11+"])
df["first_week_submissions_band"] = pd.cut(df["n_subs_first7d"], [-1, 0, 1, 5, 10, 25, 100, float("inf")],
    labels=["0", "1", "2-5", "6-10", "11-25", "26-100", "101+"])
parts=[]
for feature, col in [("team_size", "team_size_band"), ("experience", "experience_band"), ("first_week_submissions", "first_week_submissions_band")]:
    g=df.groupby(col, observed=True)["top10"].agg(teams="size", top10_teams="sum", top10_rate="mean").reset_index()
    g.insert(0,"feature",feature); g=g.rename(columns={col:"band"}); parts.append(g)
summary=pd.concat(parts,ignore_index=True)
summary.to_csv("../reports/team_success_by_band.csv",index=False,float_format="%.6f")
summary

In [ ]:
size=summary[summary.feature.eq("team_size")]
experience=summary[summary.feature.eq("experience")]
fig, axes=plt.subplots(1,2,figsize=(10,4))
axes[0].bar(size.band,size.top10_rate,color="#3182bd"); axes[0].set(title="Top-10 rate by team size",xlabel="Team size",ylabel="Share in top 10%")
axes[1].bar(experience.band,experience.top10_rate,color="#31a354"); axes[1].set(title="Top-10 rate by team experience",xlabel="Previous competitions",ylabel="Share in top 10%")
fig.tight_layout(); fig.savefig("../reports/figures/12_team_success.png",dpi=160)

Проверьте таблицу с количеством команд в каждой группе вместе с долей top-10: редкие группы могут иметь более шумную оценку. Для корректного повторения нужен кеш модели, создаваемый из реальных CSV.